# Tutorial to use ImputeGAP

## 1. Creating and visualizing an existent dataset

### 1.1. Create a timeseries object, load a series, normalize it

If I correctly understood the .txt series is a 2d representation of the dataset, each column is a channel, each row is a timestep, and then if you have multiple samples, you simply concatenate them as new rows. Then it is up to me to choose nbr_val to cut the samples.

In this case eeg-alcohol is a an individual sample, with 64 channels (features, time series) and 256 values (time length).

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset from the library
ts.load_series(utils.search_path("eeg-alcohol"))
ts.normalize(normalizer="z_score")

# print and plot a subset of time series
ts.print(nbr_series=6, nbr_val=20)
ts.plot(input_data=ts.data, nbr_series=6, nbr_val=100, save_path="./imputegap_assets")

In [ ]:
# (T, V), T: time series, V: values
ts.data.shape

### 1.2. List of all the datasets available

In [ ]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"ImputeGAP datasets : {ts.datasets}")

## 2. Contamination

In [ ]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Missingness patterns : {ts.patterns}")

## 3. Imputation

I had to install this on my ubuntu machine:
```bash
sudo apt-get install libopenblas0 libopenblas-dev
```

In [ ]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Imputation families : {ts.families}")
print(f"Imputation algorithms : {ts.algorithms}")

## 4. Others (still to check)

#### Parameter Tuning

In [ ]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"AutoML Optimizers : {ts.optimizers}")

#### Benchmark

In [ ]:
from imputegap.recovery.benchmark import Benchmark

my_algorithms = ["SoftImpute", "MeanImpute"]

my_opt = ["default_params"]

my_datasets = ["eeg-alcohol"]

my_patterns = ["mcar"]

range = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8]

my_metrics = ["*"]

# launch the evaluation
bench = Benchmark()
bench.eval(algorithms=my_algorithms, datasets=my_datasets, patterns=my_patterns, x_axis=range, metrics=my_metrics, optimizers=my_opt)

## 5. Use trained model on Physionet

In [ ]:
# ============================
# Flags for defaults
# ============================

K = 1                                 # Number of importance sampling weights
M = 1                                 # Number of samples for ELBO estimation

# banded_covar = False                  # Use banded covariance (only for gp-vae)

# basedir = "models"                    # Directory for saving models

# batch_size = 64                       # Training batch size

# cnn_kernel_size = 3                   # Kernel size of CNN preprocessor
# cnn_sizes = [256]                     # Number of filters per CNN layer (comma-separated list → list)

# data_dir = ""                         # Directory where data is read from
# data_type = "hmnist"                  # <hmnist | physionet | sprites>

# exp_name = "debug"                    # Experiment name

# gradient_clip = 10000.0               # Max global gradient norm

kernel = "cauchy"                     # GP kernel <rbf|diffusion|matern|cauchy>
kernel_scales = 1                     # Number of GP kernel length scales

# learning_rate = 0.001                 # LR

# model_type = "gp-vae"                 # <vae | hi-vae | gp-vae>

# num_steps = 0                         # Number of training steps (0 → use num_epochs)
# print_interval = 0                    # Interval for printing/saving

# seed = 1337                           # RNG seed

# testing = False                       # Whether to use the test set

#=============================
# Specific flags for Physionet experiment
#=============================
latent_dim = 35                    # override
encoder_sizes = [128, 128]         # override
decoder_sizes = [256, 256]         # override
window_size = 24                   # override
sigma = 1.005                      # override
length_scale = 7.0                 # override
beta = 0.2                         # override
num_epochs = 40                    # override
data_type = "physionet"            # override

#==============================
# Hardcoded variables
#==============================
data_dim = 35 # features in Physionet
time_length = 48 # length for a single patient (48 hours)

### 3.1 Reload a model checkpoint

In [ ]:
import os
import pandas as pd
checkpoint_dir = 'external/models/251123_reproduce_physionet'

In [ ]:
# Create the model before restoring the checkpoint
from imputegap.wrapper.AlgoPython.GPVAE.models.models import *

# read the information contained in results.tsv to build the same model used 
# during training
results = pd.read_csv(os.path.join(checkpoint_dir, "results.tsv"), delimiter="\t")

# build the model
encoder = BandedJointEncoder
decoder = GaussianDecoder
image_preprocessor = None

data_type = results.data[0]
kernel = results.kernel[0]
beta = results.beta[0]
window_size = int(results.window_size[0])
kernel_scales = results.kernel_scales[0]
sigma = results.sigma[0]
length_scale = results.length_scale[0]
encoder_sizes = [results.encoder_width[0]]*results.encoder_depth[0]
decoder_sizes = [results.decoder_width[0]]*results.decoder_depth[0]

# In the results.tsv there are some missing information that we should 
# include (TODO)
# Currently we use the ones used for the physionet training
# preprocessor needed or not?
# (cnn_kernel_size, cnn_sizes)
# latent_dim
# data_dim
# time_length
# encoder
# decoder
# image_preprocessor
# M
# K

# instantiate the model
model = GP_VAE(
    latent_dim=latent_dim, 
    data_dim=data_dim, 
    time_length=time_length,
    encoder_sizes=encoder_sizes,
    encoder=encoder,
    decoder_sizes=decoder_sizes,
    decoder=decoder,
    kernel=kernel, 
    sigma=sigma,
    length_scale=length_scale, 
    kernel_scales = kernel_scales,
    image_preprocessor=image_preprocessor, 
    window_size=window_size,
    beta=beta, 
    M=M,
    K=K,
    data_type=data_type
)

In [ ]:
# create a checkpoint with the built model
if image_preprocessor is None:
  checkpoint = tf.train.Checkpoint(
      encoder=model.encoder.net,
      decoder=model.decoder.net,
  )
else:
  checkpoint = tf.train.Checkpoint(
      encoder=model.encoder.net,
      decoder=model.decoder.net,
      preprocessor=model.preprocessor.net
  )

# restore the latest checkpoint in the checkpoint_dir
latest = tf.train.latest_checkpoint(checkpoint_dir)
checkpoint.restore(latest).expect_partial()
print("Checkpoint restored.")

### 3.2. Contaminate data and reconstruct it for an individual patient

In [ ]:
from imputegap.recovery.imputation import Imputation
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/train.txt")
ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
values_one_sample = 48
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*values_one_sample

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

# ts_m contains NaN values, I should first create a mask, and then set NaN to 0.
ts_m_mask = np.isnan(ts_m)
ts_m_compatible = np.nan_to_num(ts_m, nan=0.0)

In [ ]:
# Pass to the encoder the contaminated sample
# the model takes (B, V, T), batches, values (time steps), time series (channels)
encoder_output = model.encode([ts_m_compatible.transpose()])

In [ ]:
# Pass to the decoder the most probable latent trajectory (mean())
# and compute the most probable data space trajectory (mean())
decoder_output = model.decode(encoder_output.mean()).mean().numpy()

In [ ]:
# set the original observed data back in the imputed data (in this way
# we actually impute the original data and not simply replace the entire data
# with what the model has returned)
# decoder output is (B, V, T)
decoder_output[0].transpose()[np.invert(ts_m_mask)] = ts_m[np.invert(ts_m_mask)]

In [ ]:
# ts.data is (T, V), that's the reason we need to transpose
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=decoder_output[0].transpose(), nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(encoder_output.mean()[0, 34])
plt.show()

### Test code integration

### Create a model from zero, aim of training it

In [ ]:
from imputegap.wrapper.AlgoPython.GPVAE.models.models import *
encoder = BandedJointEncoder
decoder = GaussianDecoder
image_preprocessor = None

model = GP_VAE(
    latent_dim=latent_dim, 
    data_dim=data_dim, 
    time_length=time_length,
    encoder_sizes=encoder_sizes,
    encoder=encoder,
    decoder_sizes=decoder_sizes,
    decoder=decoder,
    kernel=kernel, 
    sigma=sigma,
    length_scale=length_scale, 
    kernel_scales = kernel_scales,
    image_preprocessor=image_preprocessor, 
    window_size=window_size,
    beta=beta, 
    M=M,
    K=K,
    data_type=data_type
)

In [ ]:
dummy_x = tf.zeros([1, time_length, data_dim])
model(dummy_x)
model.summary()
enc = encoder(latent_dim, encoder_sizes)
out = enc(dummy_x)


## 6. Integration

### 6.1 Reload a trained model

In [1]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/physionet/x_train_full.txt")
ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
seq_length = 48
nbr_features = 35
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*seq_length

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

# ts_m_imputed = gp_vae(ts_m, nbr_features, seq_length, 'external/models/251123_reproduce_physionet')
ts_m_imputed = gp_vae(ts_m, "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_physionet.yaml", 'external/models/251123_reproduce_physionet')


ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=ts_m_imputed, nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")

2025-12-04 00:40:24.570975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-04 00:40:24.593893: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-04 00:40:24.593975: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-04 00:40:24.608242: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 00:40:26.415963: W tensorflow/compiler/tf


(SYS) The dataset is loaded from ./external/datasets/physionet/x_train_full.txt

> logs: normalization (z_score) of the data - runtime: 0.0743 seconds

(CONT) missigness pattern: MCAR
	selected series: 14, 16, 20, 22, 25, 27, 30
	percentage of contaminated series: 20.0%
	rate of missing data per series: 40.0%
	block size: 10
	security offset: [0-4]
	seed value: 42

plots saved in: ./imputegap_assets/25_12_04_00_40_34_imputegap_plot.jpg


2025-12-04 00:40:38.950171: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-04 00:40:38.998495: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Checkpoint successfully restored.
(1, 48, 35)
(1, 48, 35)
[False  True]

> logs: imputation gpvae - Execution Time: 0.3894 seconds


plots saved in: ./imputegap_assets/imputation/25_12_04_00_40_41_GP-VAE_plot.jpg


'./imputegap_assets/imputation/25_12_04_00_40_41_GP-VAE_plot.jpg'

In [2]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/physionet/x_train_full.txt")
ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
seq_length = 48
nbr_features = 35
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*seq_length

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

# ts_m_imputed = gp_vae(ts_m, nbr_features, seq_length, 'external/models/251123_reproduce_physionet')
ts_m_imputed = gp_vae(ts_m, "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_physionet.yaml", 'imputegap_assets/models/exp_test')


ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=ts_m_imputed, nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")


(SYS) The dataset is loaded from ./external/datasets/physionet/x_train_full.txt

> logs: normalization (z_score) of the data - runtime: 0.0733 seconds

(CONT) missigness pattern: MCAR
	selected series: 14, 16, 20, 22, 25, 27, 30
	percentage of contaminated series: 20.0%
	rate of missing data per series: 40.0%
	block size: 10
	security offset: [0-4]
	seed value: 42

plots saved in: ./imputegap_assets/25_12_04_00_42_20_imputegap_plot.jpg
Checkpoint successfully restored.
(1, 48, 35)
(1, 48, 35)
[False  True]

> logs: imputation gpvae - Execution Time: 0.1204 seconds


plots saved in: ./imputegap_assets/imputation/25_12_04_00_42_25_GP-VAE_plot.jpg


'./imputegap_assets/imputation/25_12_04_00_42_25_GP-VAE_plot.jpg'

### 6.2 Train a model

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/physionet/x_miss_nan.txt")
# ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
seq_length = 48
nbr_features = 35
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*seq_length

# Add missingness to the data, ts has shape (T, V)
# ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

# ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

# ts_m_imputed = gp_vae(ts_m, nbr_features, seq_length, 'external/models/251123_reproduce_physionet')
ts_m_imputed = gp_vae(ts.data, "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_physionet.yaml", batch_size=32, epoch=4, verbose=False)


# ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=ts_m_imputed, nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")

2025-12-04 00:35:48.890818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-04 00:35:48.911256: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-04 00:35:48.911283: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-04 00:35:48.923894: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 00:35:50.725016: W tensorflow/compiler/tf


(SYS) The dataset is loaded from ./external/datasets/physionet/x_miss_nan.txt

Start model training...


2025-12-04 00:36:04.896736: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-04 00:36:04.935623: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
Training progress:   0%|          | 0/499 [00:00<?, ?it/s]

Instructions for updating:
`MultivariateNormalFullCovariance` is deprecated, use `MultivariateNormalTriL(loc=loc, scale_tril=tf.linalg.cholesky(covariance_matrix))` instead.
Learning rate: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513> | Global gradient norm: 0.00
Step 0) Time = 0.611906
Train loss = -0.000 | NLL = 0.000 | KL = 0.000


Training progress:   0%|          | 1/499 [00:01<08:18,  1.00s/it]

Validation loss = -0.000 | NLL = 0.000 | KL = 0.000


Training progress:   4%|▍         | 22/499 [00:08<02:58,  2.67it/s]


Finished training
(11987, 48, 35)


2025-12-04 00:36:18.305006: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1275416800 exceeds 10% of free system memory.
2025-12-04 00:36:18.993987: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1275416800 exceeds 10% of free system memory.
2025-12-04 00:36:24.774865: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 3866526720 exceeds 10% of free system memory.


In [ ]:
import numpy as np
import os
np.loadtxt("external/datasets/physionet/m_miss.txt").transpose().reshape(-1, 48, 35).shape

In [ ]:
np.loadtxt("external/datasets/physionet/x_train_full.txt").transpose().reshape(-1, 48, 35).shape

In [ ]:
np.loadtxt("external/datasets/physionet/x_val_full.txt").transpose().reshape(-1, 48, 35).shape

In [ ]:
np.loadtxt("external/datasets/physionet/x_test_full.txt").transpose().reshape(-1, 48, 35).shape

In [ ]:
191856*2 + 191664